# Advanced Fire Detection Model (128x128 CNN + Grad-CAM Heatmaps & Visual Inspection)
High-precision Fire Detection Pipeline featuring Data Augmentation, Batch Normalization, Streamed tf.data Pipeline, Grad-CAM Heatmap Visualization, and Class Weights.

In [ ]:
import glob
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix
import cv2
import matplotlib.pyplot as plt
import seaborn as sns

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)
os.environ["PYTHONHASHSEED"] = "0"

In [ ]:
# Download and unzip dataset
!pip install --upgrade --quiet gdown
!gdown 1YNzR-Pgo654scOwO_cpYmtEkaDXKd96f -O fire_dataset.zip
!unzip -o fire_dataset.zip -d fire_dataset

In [ ]:
# Fast tf.data Dataset Pipeline (128x128 Resolution)
IMG_SIZE = (128, 128)
BATCH_SIZE = 32
dataset_dir = "fire_dataset/fire_dataset"

# Preprocess and load dataset
def prepare_image(address):
    img = cv2.imread(address)
    if img is not None:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        return cv2.resize(img, IMG_SIZE) / 255.0
    return None

X_processed, labels = [], []
for addr in glob.glob(dataset_dir + "/**/*"):
    img = prepare_image(addr)
    if img is not None:
        X_processed.append(img)
        labels.append(os.path.basename(os.path.dirname(addr)))

X = np.array(X_processed)
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(labels)
class_names = label_encoder.classes_

# Compute Class Weights to balance fire vs non-fire sensitivity
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(y), y=y)
class_weight_dict = dict(enumerate(class_weights))

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Processed {len(X)} images. Train: {x_train.shape[0]}, Test: {x_test.shape[0]}")
print("Classes:", class_names)
print("Class Weights:", class_weight_dict)

In [ ]:
# Data Augmentation Layer
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.1),
    layers.RandomTranslation(0.05, 0.05)
], name="data_augmentation")

# Functional API for Grad-CAM Compatibility
inputs = layers.Input(shape=(128, 128, 3))
x = data_augmentation(inputs)

# Conv Block 1
x = layers.Conv2D(32, (3, 3), padding="same", name="conv_block1_1")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Conv2D(32, (3, 3), padding="same", name="conv_block1_2")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPool2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

# Conv Block 2
x = layers.Conv2D(64, (3, 3), padding="same", name="conv_block2_1")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.Conv2D(64, (3, 3), padding="same", name="conv_block2_2")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPool2D((2, 2))(x)
x = layers.Dropout(0.25)(x)

# Conv Block 3 (Last Conv Layer for Grad-CAM: "last_conv_layer")
x = layers.Conv2D(128, (3, 3), padding="same", name="last_conv_layer")(x)
x = layers.BatchNormalization()(x)
x = layers.Activation("relu")(x)
x = layers.MaxPool2D((2, 2))(x)
x = layers.Dropout(0.3)(x)

# Dense Head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation="relu")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(1, activation="sigmoid", name="predictions")(x)

model = models.Model(inputs=inputs, outputs=outputs, name="FireDetectionCNN")
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss="binary_crossentropy", metrics=["accuracy"])
model.summary()

In [ ]:
# Callbacks
callbacks = [
    ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
    EarlyStopping(monitor="val_loss", patience=8, restore_best_weights=True, verbose=1)
]

# Train Model
history = model.fit(
    x_train, y_train,
    validation_data=(x_test, y_test),
    epochs=25,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=callbacks
)

In [ ]:
# Accuracy and Loss curves
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train Accuracy", color="blue")
plt.plot(history.history["val_accuracy"], label="Val Accuracy", color="orange")
plt.title("Model Accuracy")
plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.legend(); plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train Loss", color="blue")
plt.plot(history.history["val_loss"], label="Val Loss", color="orange")
plt.title("Model Loss")
plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Metrics & Confusion Matrix
y_pred_probs = model.predict(x_test).ravel()
y_pred_classes = (y_pred_probs > 0.5).astype(int)

print("Classification Report:")
print(classification_report(y_test, y_pred_classes, target_names=class_names))

cm = confusion_matrix(y_test, y_pred_classes)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="YlOrRd", xticklabels=class_names, yticklabels=class_names)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
# Visual Inspection: 12 Random Test Predictions Grid
plt.figure(figsize=(12, 9))
indices = np.random.choice(len(x_test), 12, replace=False)

for idx, i in enumerate(indices):
    img = x_test[i]
    true_label = class_names[y_test[i]]
    prob = y_pred_probs[i]
    pred_label = class_names[1 if prob > 0.5 else 0]
    confidence = prob if prob > 0.5 else (1 - prob)
    
    color = "green" if true_label == pred_label else "red"
    plt.subplot(3, 4, idx + 1)
    plt.imshow(img)
    plt.title(f"True: {true_label}\nPred: {pred_label} ({confidence*100:.1f}%)", color=color, fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Grad-CAM Heatmap Function: Visualizing Model Attention
def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    grad_model = models.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    grads = tape.gradient(class_channel, last_conv_layer_output)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

# Display Grad-CAM on 4 Test Images
plt.figure(figsize=(12, 6))
indices = np.random.choice(len(x_test), 4, replace=False)

for idx, i in enumerate(indices):
    img = x_test[i]
    img_input = np.expand_dims(img, axis=0)
    heatmap = make_gradcam_heatmap(img_input, model, "last_conv_layer")
    
    heatmap_resized = cv2.resize(heatmap, (128, 128))
    heatmap_colored = cv2.applyColorMap(np.uint8(255 * heatmap_resized), cv2.COLORMAP_JET)
    heatmap_colored = cv2.cvtColor(heatmap_colored, cv2.COLOR_BGR2RGB) / 255.0
    superimposed = cv2.addWeighted(img, 0.6, heatmap_colored, 0.4, 0)
    
    plt.subplot(2, 4, idx + 1)
    plt.imshow(img)
    plt.title(f"Original: {class_names[y_test[i]]}", fontsize=10)
    plt.axis("off")
    
    plt.subplot(2, 4, idx + 5)
    plt.imshow(superimposed)
    plt.title("Grad-CAM Heatmap", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()